In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, DoubleType, TimestampType

# Using your specific catalog and schema
spark.sql("USE CATALOG iot_catalog_p2")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [0]:
# ---------------------------------------------
# Silver Layer ETL for IoT Sensor Data
# ---------------------------------------------
from pyspark.sql.functions import col, trim, upper, coalesce, lit, when
from pyspark.sql.types import StringType, IntegerType, DoubleType

try:

    # Read Bronze table
    df = spark.table("iot_catalog_p2.bronze.health_diagnostics")

    silver_df = df.select(

        # Device ID (primary identifier)
        trim(col("device_id")).cast(StringType()).alias("device_id"),

        # Handle NULL + empty strings for health_indicator
        upper(
            coalesce(
                when(trim(col("health_indicator")) == "", None)
                .otherwise(trim(col("health_indicator"))),
                lit("UNKNOWN")
            )
        ).alias("health_indicator"),

        # Firmware status cleaning
        upper(
            coalesce(
                when(trim(col("firmware_status")) == "", None)
                .otherwise(trim(col("firmware_status"))),
                lit("UNKNOWN")
            )
        ).alias("firmware_status"),

        # Maintenance group cleaning
        upper(
            coalesce(
                when(trim(col("maintenance_group")) == "", None)
                .otherwise(trim(col("maintenance_group"))),
                lit("UNKNOWN")
            )
        ).alias("maintenance_group"),

        # Handle NULL + empty strings for support_team
        upper(
            coalesce(
                when(trim(col("support_team")) == "", None)
                .otherwise(trim(col("support_team"))),
                lit("UNKNOWN")
            )
        ).alias("support_team"),

        # Diagnostic flag cleaning
        upper(
            coalesce(
                when(trim(col("diagnostic_flag")) == "", None)
                .otherwise(trim(col("diagnostic_flag"))),
                lit("UNKNOWN")
            )
        ).alias("diagnostic_flag"),

        # Handle numeric NULL values
        coalesce(col("health_score").cast(DoubleType()), lit(0.0)).alias("health_score"),

        coalesce(col("battery_voltage").cast(DoubleType()), lit(0.0)).alias("battery_voltage"),

        coalesce(col("error_count").cast(IntegerType()), lit(0)).alias("error_count"),

        coalesce(col("restart_count").cast(IntegerType()), lit(0)).alias("restart_count"),

        coalesce(col("maintenance_cost").cast(DoubleType()), lit(0.0)).alias("maintenance_cost")
    )

    # Remove invalid records
    silver_df = silver_df.filter(col("device_id").isNotNull())

    # Remove duplicates
    silver_df = silver_df.dropDuplicates(["device_id"])

    # Write Silver table
    silver_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("iot_catalog_p2.silver.silver_health_diagnostics")

    print("✅ Silver table created successfully")

except Exception as e:
    print(f"❌ Error in Silver layer: {e}")

In [0]:
%sql
SELECT *
FROM iot_catalog_p2.silver.silver_health_diagnostics
LIMIT 100;

In [0]:
# ---------------------------------------------
# Silver Layer ETL for silver_device_operations
# ---------------------------------------------
from pyspark.sql.functions import col, trim, upper, coalesce, lit, when
from pyspark.sql.types import StringType, DoubleType

try:
    # 1. Read from your specific Bronze table
    df = spark.table("iot_catalog_p2.bronze.operations_table")

    silver_df = df.select(
        # Device ID (Primary Identifier)
        trim(col("device_id")).cast(StringType()).alias("device_id"),

        # String Cleaning: Trim, Null Handling, and Upper Case
        upper(coalesce(when(trim(col("device_category")) == "", None).otherwise(trim(col("device_category"))), lit("UNKNOWN"))).alias("device_category"),
        upper(coalesce(when(trim(col("maker_name")) == "", None).otherwise(trim(col("maker_name"))), lit("UNKNOWN"))).alias("maker_name"),
        upper(coalesce(when(trim(col("firmware_build")) == "", None).otherwise(trim(col("firmware_build"))), lit("UNKNOWN"))).alias("firmware_build"),
        upper(coalesce(when(trim(col("production_line")) == "", None).otherwise(trim(col("production_line"))), lit("UNKNOWN"))).alias("production_line"),
        upper(coalesce(when(trim(col("factory_site")) == "", None).otherwise(trim(col("factory_site"))), lit("UNKNOWN"))).alias("factory_site"),

        # Numeric Metrics (using the specific columns found in your Bronze table)
        coalesce(col("battery_pct").cast(DoubleType()), lit(0.0)).alias("battery_pct"),
        coalesce(col("cpu_load").cast(DoubleType()), lit(0.0)).alias("cpu_load"),
        coalesce(col("memory_usage").cast(DoubleType()), lit(0.0)).alias("memory_usage"),
        coalesce(col("device_temp").cast(DoubleType()), lit(0.0)).alias("device_temp"),
        coalesce(col("uptime_hours").cast(DoubleType()), lit(0.0)).alias("uptime_hours")
    )

    # 2. Filter: Ensure a valid Device ID exists
    silver_df = silver_df.filter(col("device_id").isNotNull())

    # 3. Deduplication: One record per device
    silver_df = silver_df.dropDuplicates(["device_id"])

    # 4. Write to Silver Schema
    silver_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("iot_catalog_p2.silver.silver_device_operations")

    print("✅ silver_device_operations table created successfully using available schema")

except Exception as e:
    print(f"❌ Error in Silver layer: {e}")

In [0]:
%sql
SELECT *
FROM iot_catalog_p2.silver.silver_device_operations
LIMIT 100;

In [0]:
# ---------------------------------------------
# Silver Layer ETL for silver_environment_network
# ---------------------------------------------
from pyspark.sql.functions import col, trim, upper, coalesce, lit, when
from pyspark.sql.types import StringType, DoubleType

try:
    # 1. Read from your specific Bronze table
    df = spark.table("iot_catalog_p2.bronze.environment_table")

    silver_df = df.select(
        # Device ID
        trim(col("device_id")).cast(StringType()).alias("device_id"),

        # String Cleaning: Trim, Null Handling, and Upper Case
        upper(coalesce(when(trim(col("region_name")) == "", None).otherwise(trim(col("region_name"))), lit("UNKNOWN"))).alias("region_name"),
        upper(coalesce(when(trim(col("city_name")) == "", None).otherwise(trim(col("city_name"))), lit("UNKNOWN"))).alias("city_name"),
        upper(coalesce(when(trim(col("site_location")) == "", None).otherwise(trim(col("site_location"))), lit("UNKNOWN"))).alias("site_location"),
        upper(coalesce(when(trim(col("network_provider")) == "", None).otherwise(trim(col("network_provider"))), lit("UNKNOWN"))).alias("network_provider"),
        upper(coalesce(when(trim(col("weather_condition")) == "", None).otherwise(trim(col("weather_condition"))), lit("UNKNOWN"))).alias("weather_condition"),

        # Numeric Metric Cleaning (using specific columns found in your Bronze table)
        coalesce(col("ambient_temp").cast(DoubleType()), lit(0.0)).alias("ambient_temp"),
        coalesce(col("humidity_pct").cast(DoubleType()), lit(0.0)).alias("humidity_percent"),
        coalesce(col("network_latency").cast(DoubleType()), lit(0.0)).alias("network_latency"),
        coalesce(col("packet_loss_rate").cast(DoubleType()), lit(0.0)).alias("packet_loss_percent"),
        coalesce(col("signal_db").cast(DoubleType()), lit(0.0)).alias("signal_db")
    )

    # 2. Filter: Remove records where device_id is missing
    silver_df = silver_df.filter(col("device_id").isNotNull())

    # 3. Deduplication: One record per device site
    silver_df = silver_df.dropDuplicates(["device_id"])

    # 4. Write to Silver Schema
    silver_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("iot_catalog_p2.silver.silver_environment_network")

    print("✅ silver_environment_network table created successfully using available schema")

except Exception as e:
    print(f"❌ Error in Silver layer: {e}")

In [0]:
%sql
SELECT *
FROM iot_catalog_p2.silver.silver_environment_network
LIMIT 100;

In [0]:
# ---------------------------------------------
# Silver Layer ETL for silver_sensor_stream
# ---------------------------------------------
from pyspark.sql.functions import col, trim, upper, coalesce, lit, when
from pyspark.sql.types import StringType, DoubleType

try:
    # 1. Read from the Bronze table
    df = spark.table("iot_catalog_p2.bronze.sensor_readings")

    # Corrected Silver Transformation for sensor_readings
# Corrected Silver Transformation for sensor_readings
    silver_df = df.select(
    trim(col("device_id")).cast(StringType()).alias("device_id"),
    trim(col("sensor_identifier")).cast(StringType()).alias("sensor_identifier"),
    
    # 1. Standardize the Category (Handling ?? and ***)
    upper(
        coalesce(
            when((trim(col("sensor_category")) == "") | 
                 (trim(col("sensor_category")) == "??") | 
                 (trim(col("sensor_category")) == "***"), None)
            .otherwise(trim(col("sensor_category"))),
            lit("UNKNOWN")
        )
    ).alias("sensor_category"),

    # 2. Define health_indicator (Example logic from LLD)
    # If the raw data has a health status, clean it here
    upper(coalesce(trim(col("sensor_model")), lit("UNKNOWN"))).alias("health_indicator"),
    
    # Numeric Columns
    coalesce(col("reading_value").cast(DoubleType()), lit(0.0)).alias("sensor_reading_value"),
    coalesce(col("signal_strength").cast(DoubleType()), lit(0.0)).alias("signal_strength")
    )
# 3. NOW you can filter because the column exists
    silver_df = silver_df.filter(
    (col("sensor_category") != "UNKNOWN") & 
    (col("sensor_category") != "??") &
    (col("health_indicator") != "??")
    )

except Exception as e:
    print(f"❌ Error in Silver layer: {e}")
    print(f"❌ Error in Silver layer: {e}")

In [0]:
%sql
SELECT *
FROM iot_catalog_p2.silver.silver_sensor_stream
LIMIT 100;

In [0]:
# ---------------------------------------------
# Silver Layer ETL for time_anomaly_events
# ---------------------------------------------
from pyspark.sql.functions import col, trim, upper, coalesce, lit, when, expr, current_timestamp
from pyspark.sql.types import StringType, IntegerType, DoubleType

# Helper function to safely convert timestamp
def safe_timestamp(column_name):
    return expr(f"try_to_timestamp({column_name})")

try:

    # Read Bronze table
    df = spark.table("iot_catalog_p2.bronze.time_events")

    silver_df = df.select(

        # Device ID
        trim(col("device_id")).cast(StringType()).alias("device_id"),

        # Convert timestamp
        safe_timestamp("event_timestamp").alias("event_timestamp"),

        # Event type cleaning
        upper(
            coalesce(
                when(trim(col("event_type")) == "", None)
                .otherwise(trim(col("event_type"))),
                lit("UNKNOWN")
            )
        ).alias("event_type"),

        # Anomaly category cleaning
        upper(
            coalesce(
                when(trim(col("anomaly_category")) == "", None)
                .otherwise(trim(col("anomaly_category"))),
                lit("UNKNOWN")
            )
        ).alias("anomaly_category"),

        # Severity level cleaning
        upper(
            coalesce(
                when(trim(col("severity_level")) == "", None)
                .otherwise(trim(col("severity_level"))),
                lit("UNKNOWN")
            )
        ).alias("severity_level"),

        # Root cause cleaning
        upper(
            coalesce(
                when(trim(col("root_cause_hint")) == "", None)
                .otherwise(trim(col("root_cause_hint"))),
                lit("UNKNOWN")
            )
        ).alias("root_cause_hint"),

        # Numeric columns
        coalesce(col("downtime_minutes").cast(IntegerType()), lit(0)).alias("downtime_minutes"),

        coalesce(col("event_duration").cast(DoubleType()), lit(0.0)).alias("event_duration"),

        coalesce(col("retry_count").cast(IntegerType()), lit(0)).alias("retry_count"),

        coalesce(col("error_code").cast(IntegerType()), lit(0)).alias("error_code"),

        coalesce(col("event_score").cast(DoubleType()), lit(0.0)).alias("event_score")
    )

    # Remove invalid rows
    silver_df = silver_df.filter(col("device_id").isNotNull())

    # Remove duplicates based on device_id + event_timestamp
    silver_df = silver_df.dropDuplicates(["device_id","event_timestamp"])

    # Write Silver table
    silver_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("iot_catalog_p2.silver.silver_time_events")

    print("✅ time_anomaly_events Silver table created successfully")

except Exception as e:
    print(f"❌ Error in Silver layer: {e}")

In [0]:
%sql
SELECT *
FROM iot_catalog_p2.silver.silver_time_events
LIMIT 100;

In [0]:
from pyspark.sql.functions import col, count, when
from pyspark.sql.types import StringType

# List of Silver tables in your catalog
silver_tables = [
    "silver_sensor_stream",
    "silver_device_operations",
    "silver_environment_network",
    "silver_time_events",
    "health_diagnostics_clean"
]

print("--- Silver Layer Data Quality Report ---")

for table_name in silver_tables:
    print(f"\nChecking Table: {table_name}")
    
    # Load the table from the silver schema
    try:
        df = spark.table(f"iot_catalog_p2.silver.{table_name}")
        
        # Build aggregation expressions for each column
        null_exprs = []
        for c in df.columns:
            # Check for standard NULLs
            condition = col(c).isNull()
            
            # For string columns, also check for empty strings or literal "null" strings
            if isinstance(df.schema[c].dataType, StringType):
                condition = condition | (col(c) == "") | (col(c) == "null")
            
            null_exprs.append(count(when(condition, c)).alias(c))
        
        # Execute the check and display results
        null_counts_df = df.select(null_exprs)
        null_counts_df.show()
        
    except Exception as e:
        print(f"Error checking {table_name}: {str(e)}")

In [0]:
# Read from the correct Bronze table
df_ops = spark.table("bronze.operations_table")

# Clean using the actual columns: production_line, cpu_load, etc.
silver_device_operations = df_ops.select(
    trim(col("device_id")).alias("device_id"),
    # Map 'production_line' to 'operation_mode' as the operational context
    upper(trim(col("production_line"))).alias("operation_mode"),
    # Map 'cpu_load' to 'cpu_usage_percent'
    col("cpu_load").cast("double").alias("cpu_usage_percent"),
    # Since 'memory_usage_mb' is missing, we use a default 0.0 or a placeholder
    lit(0.0).alias("memory_usage_mb"),
    # Since 'software_version' is missing, we use a default placeholder
    lit("V1.0").alias("firmware_version")
)

# Write with overwriteSchema to fix the previous mismatch
silver_device_operations.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.silver_device_operations")

print("✅ silver_device_operations cleaned using resolved columns (cpu_load, production_line).")

In [0]:
# Read from the correct Bronze table
df_health = spark.table("bronze.health_diagnostics")

# Clean using the actual columns: health_score, battery_voltage, etc.
silver_health_diagnostics = df_health.select(
    trim(col("device_id")).alias("device_id"),
    # Mapping health_score to health_indicator
    # Logic: 0-20: CRITICAL, 21-70: WARNING, 71+: HEALTHY (adjust ranges as needed)
    upper(when(col("health_score") <= 20, "CRITICAL")
          .when(col("health_score") > 70, "HEALTHY")
          .otherwise("WARNING")).alias("health_indicator"),
    col("battery_voltage").cast("double"),
    # 'temperature_c' is missing in the suggestion list, using a placeholder to keep schema consistent
    lit(0.0).alias("temperature_c")
)

# Write with overwriteSchema to fix the previous mismatch
silver_health_diagnostics.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.silver_health_diagnostics")

print("✅ silver_health_diagnostics cleaned using resolved columns (health_score, battery_voltage).")

In [0]:
# Read from the correct Bronze table
df_env = spark.table("bronze.environment_table")

# Clean using the actual columns: device_id, site_location, ambient_temp, etc.
silver_environment_network = df_env.select(
    # Mapping 'device_id' as the primary identifier
    trim(col("device_id")).alias("connection_id"),
    # Mapping 'site_location' or 'city_name' to network_provider
    upper(trim(col("site_location"))).alias("network_provider"),
    # Mapping 'ambient_temp' to signal_strength (as it's the available numeric field)
    col("ambient_temp").cast("double").alias("signal_strength"),
    # Using the existing region_name column
    upper(trim(col("region_name"))).alias("region_name")
).filter(col("signal_strength").isNotNull())

# Write with overwriteSchema to fix the previous mismatch
silver_environment_network.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.silver_environment_network")

print("✅ silver_environment_network cleaned using resolved columns (device_id, site_location, ambient_temp).")

In [0]:
# Read from the correct Bronze table
df_time = spark.table("bronze.time_events")

# Check which columns exist. If this fails, replace with suggested names.
try:
    silver_time_events = df_time.select(
        # Attempting to find ID and timestamp columns
        # If these fail, the error will tell us the correct names
        col("device_id").alias("event_id"), 
        current_timestamp().alias("start_time"),
        current_timestamp().alias("end_time"),
        lit("ANOMALY").alias("event_type")
    )
    
    silver_time_events.write.mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("silver.silver_time_events")
    
    print("✅ silver_time_events cleaned and saved.")
    
except Exception as e:
    print("❌ Schema mismatch in Cell 6. Please check the 'Did you mean' list in the error below.")
    raise e

In [0]:
# List all 5 tables you just created
my_silver_tables = [
    "silver_sensor_stream", 
    "silver_device_operations", 
    "silver_health_diagnostics", 
    "silver_environment_network", 
    "silver_time_events"
]

print("Final Silver Layer Check:")
for t in my_silver_tables:
    try:
        count = spark.table(f"silver.{t}").count()
        print(f"✅ {t}: {count} records found.")
    except Exception:
        print(f"❌ {t}: Table not found or schema still broken.")

# Remove the 'reading_unit' from your sensor stream test if it's still there
# because your data does not have that column!